# push_t_book_3 — CEM planning visuals

Book versions of the P75 CEM-planning sequence (`p75b` manim scene).

Two halves:
- **Visualize** (sections 1–4): needs only the saved run data (`iterations.npz`, `config.json`) that
  `p75bb.ipynb` wrote — no model, no env. Start/goal frames are vector-rendered from the states in
  `config.json`; candidate paths are drawn as line collections in the same 512-px world coordinates.
- **Regenerate** (section 5, optional): re-run the instrumented CEM for a different episode / config
  and write the same files. Requires the model + dataset.

Data layout (from `p75bb`):
```
iterations.npz   paths_all (n_steps, N, N_ACTS, 2)  image px (224), y down
                 paths_elite (n_steps, K, N_ACTS, 2)   paths_mean (n_steps, N_ACTS, 2)
                 costs (n_steps, N)   topk_inds (n_steps, K)   cum_sigma (n_steps, N_ACTS, 2)
config.json      agent_xy_img, start_state, goal_state, scale (224/512), CEM config …
```


## 1. Config

In [ ]:
from pathlib import Path

# ── Print palette ──────────────────────────────────────────────────────────
CHILL_BROWN = "#948678"
LT_BLUE     = "#5badb6"
ORANGE      = "#eb8423"
FRESH_TAN   = "#f4ebd9"
DEEP_TEAL   = "#063336"      # dark background for the CEM figures
YELLOW      = "#ffd35a"
MAGENTA     = "#ff00ff"

PALETTE = dict(               # the tan "game frame" look (book_1 / book_2)
    target=CHILL_BROWN, movable=LT_BLUE, effector=ORANGE, background=FRESH_TAN,
    outline=None, frame=CHILL_BROWN, frame_width=1.5,
)
CEM_PALETTE = dict(PALETTE,   # same shapes on the dark CEM background — override freely
    background=DEEP_TEAL,
    frame=CHILL_BROWN,
)

# ── CEM run to visualize ───────────────────────────────────────────────────
HACKIN   = Path("/home/stephen/Stephencwelch Dropbox/welch_labs/jepa_2/hackin")
RUN_DIR  = HACKIN / "p75b" / "ep2167_off25_h5"     # written by p75bb (or section 5 below)
OUT_DIR  = HACKIN / "push_t_book_3" / RUN_DIR.name

# ── Path drawing (first viz: all candidates at iteration 0) ────────────────
STEP          = 0           # CEM iteration to draw
N_PATHS       = 500         # how many of the N candidates to draw
PATH_COLOR    = MAGENTA
PATH_ALPHA    = 0.20
PATH_WIDTH    = 0.6         # points
START_DIM     = 0.15        # opacity of the start-frame shapes under the paths (manim used 0.1)
SHOW_START_DOT = True       # dot at the effector start position
START_DOT_COLOR = ORANGE

# ── Export ─────────────────────────────────────────────────────────────────
EXPORT_TRANSPARENT = True   # True: no background rect in the SVG (notebook preview still shows DEEP_TEAL)
FIG_SIZE = 4.0              # inches

import matplotlib.pyplot as plt
plt.rcParams["svg.fonttype"] = "none"


## 2. Load the saved CEM run

In [ ]:
import json
import numpy as np
from matplotlib.patches import Polygon, Circle, Rectangle
from matplotlib.collections import LineCollection

d   = np.load(RUN_DIR / "iterations.npz")
cfg = json.load(open(RUN_DIR / "config.json"))
for k in d.files: print(f"{k:12s} {d[k].shape}")

IMG_SIZE = cfg["img_size"]           # 224 — coordinate space of the saved paths
WORLD    = cfg["pusht_native"]       # 512 — coordinate space of env states / our renderer
IMG2WORLD = WORLD / IMG_SIZE

start_state  = np.array(cfg["start_state"])      # (7,)
goal_state   = np.array(cfg["goal_state"])       # (7,)
agent_xy     = np.array(cfg["agent_xy_img"]) * IMG2WORLD     # effector start, world px
N_STEPS, N, N_ACTS, _ = d["paths_all"].shape
K = d["paths_elite"].shape[1]

def paths_world(step, which="all", n=None):
    """(M, N_ACTS+1, 2) world-px polylines for one CEM iteration, start point prepended."""
    p = d[f"paths_{which}"][step]
    if p.ndim == 2: p = p[None]                     # paths_mean is (N_ACTS, 2)
    if n is not None: p = p[:n]
    p = p * IMG2WORLD
    start = np.broadcast_to(agent_xy, (p.shape[0], 1, 2))
    return np.concatenate([start, p], axis=1)

print(f"\nepisode {cfg['episode_idx']}  start t={cfg['start_step']}  goal +{cfg['goal_offset']}   "
      f"{N_STEPS} iters × {N} samples × {N_ACTS} actions, top-{K}")
print("effector start (world px):", agent_xy.round(1))


## 3. Renderer (vector shapes + path collections)

In [ ]:
# ── PushT geometry (stable_worldmodel/envs/pusht/env.py) ───────────────────
GOAL_POSE    = np.array([256.0, 256.0, np.pi / 4])
AGENT_RADIUS = 0.375 * 40
_S, _L = 30, 120
_T_OUTLINE = np.array([(-_L/2, 0), (_L/2, 0), (_L/2, _S), (_S/2, _S), (_S/2, _L),
                       (-_S/2, _L), (-_S/2, _S), (-_L/2, _S)], float)

def tee_world_vertices(x, y, angle):
    c, s = np.cos(angle), np.sin(angle)
    return (_T_OUTLINE @ np.array([[c, -s], [s, c]]).T) + (x, y)

def draw_tee(ax, pose, color, alpha=1.0, zorder=1):
    ax.add_patch(Polygon(tee_world_vertices(*pose), closed=True, facecolor=color,
                         edgecolor=color, linewidth=0, alpha=alpha, joinstyle="miter", zorder=zorder))

def setup_axes(ax, palette, draw_bg=True):
    """Transparent axes patch, optional background square, frame box on top, 512-world limits."""
    ax.patch.set_alpha(0)
    if draw_bg:
        ax.add_patch(Rectangle((0, 0), WORLD, WORLD, facecolor=palette["background"],
                               edgecolor="none", zorder=0))
    if palette.get("frame"):
        ax.add_patch(Rectangle((0, 0), WORLD, WORLD, facecolor="none", edgecolor=palette["frame"],
                               linewidth=palette.get("frame_width", 1.5), joinstyle="miter",
                               clip_on=False, zorder=10))
    ax.set_xlim(0, WORLD); ax.set_ylim(WORLD, 0); ax.set_aspect("equal"); ax.axis("off")
    return ax

def draw_state(ax, state, palette, alpha=1.0, show_goal=True, show_effector=True, zorder=1):
    """Shapes only (no background) so they can be layered under/over paths with their own alpha."""
    if show_goal:
        draw_tee(ax, GOAL_POSE, palette["target"], alpha, zorder)
    draw_tee(ax, (state[2], state[3], state[4]), palette["movable"], alpha, zorder + 0.1)
    if show_effector:
        ax.add_patch(Circle((state[0], state[1]), AGENT_RADIUS, facecolor=palette["effector"],
                            edgecolor="none", alpha=alpha, zorder=zorder + 0.2))

def draw_paths(ax, segs, color=PATH_COLOR, alpha=PATH_ALPHA, width=PATH_WIDTH, zorder=3, **kw):
    """segs: (M, P, 2) polylines in world px. color may be one color or a list/array per path."""
    lc = LineCollection(segs, colors=color, linewidths=width, alpha=alpha,
                        capstyle="round", joinstyle="round", zorder=zorder, **kw)
    ax.add_collection(lc)
    return lc

def new_figure(size=FIG_SIZE, palette=CEM_PALETTE, preview_bg=True):
    """Square figure; the figure facecolor shows the background in the notebook even when the
    exported SVG is transparent."""
    fig = plt.figure(figsize=(size, size))
    fig.patch.set_facecolor(palette["background"] if preview_bg else "none")
    ax = fig.add_axes([0.02, 0.02, 0.96, 0.96])
    return fig, ax

def save_svg(fig, path, transparent=EXPORT_TRANSPARENT):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, transparent=transparent, bbox_inches="tight", pad_inches=0.02)
    print("saved", path)


### Sanity check: start + goal frames from `config.json`
Left: start (what the planner sees). Right: goal frame. Compare against `RUN_DIR/start_img.png` / `goal_img.png`.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3.5))
for ax, st, title in zip(axs, [start_state, goal_state], ["start", "goal"]):
    setup_axes(ax, PALETTE); draw_state(ax, st, PALETTE); ax.set_title(title)
plt.tight_layout(); plt.show()


## 4. Viz 1 — all candidate paths at iteration `STEP` over the start frame

Knobs live in the config cell (`PATH_COLOR`, `PATH_ALPHA`, `PATH_WIDTH`, `START_DIM`, …) but you can
also override them inline here while noodling.

In [ ]:
segs = paths_world(STEP, "all", N_PATHS)                 # (N_PATHS, 26, 2)

fig, ax = new_figure()
setup_axes(ax, CEM_PALETTE, draw_bg=not EXPORT_TRANSPARENT)
draw_state(ax, start_state, CEM_PALETTE, alpha=START_DIM)
draw_paths(ax, segs, color=PATH_COLOR, alpha=PATH_ALPHA, width=PATH_WIDTH)
if SHOW_START_DOT:
    ax.add_patch(Circle(agent_xy, 6, facecolor=START_DOT_COLOR, edgecolor="none", zorder=5))

save_svg(fig, OUT_DIR / f"viz1_paths_step{STEP:02d}_n{N_PATHS}.svg")
plt.show()


### Quick variations
A small grid to compare alpha / width / color choices side by side before committing.

In [ ]:
variants = [
    dict(color=MAGENTA,  alpha=0.10, width=0.6),
    dict(color=MAGENTA,  alpha=0.25, width=0.6),
    dict(color=YELLOW,   alpha=0.15, width=0.5),
    dict(color=LT_BLUE,  alpha=0.20, width=0.8),
]
fig, axs = plt.subplots(1, len(variants), figsize=(3.2 * len(variants), 3.2))
fig.patch.set_facecolor(CEM_PALETTE["background"])
for ax, v in zip(axs, variants):
    setup_axes(ax, CEM_PALETTE, draw_bg=True)
    draw_state(ax, start_state, CEM_PALETTE, alpha=START_DIM)
    draw_paths(ax, segs, **v)
    ax.set_title(f"{v['color']}  α={v['alpha']}  w={v['width']}", fontsize=8, color=FRESH_TAN)
plt.show()


---
## 5. (Optional) Regenerate CEM data for another episode / config

Mirrors `p75bb` sections 1–2: builds the world at a dataset episode, runs the instrumented CEM solver,
projects candidates to image px, and writes `iterations.npz` + `config.json` + `start_img.png` /
`goal_img.png` into a new `RUN_DIR`. Then set `RUN_DIR` in the config cell to it and re-run from section 2.
Needs the model + dataset (same env as `push_t_book_2`).

In [ ]:
import warnings, time
warnings.filterwarnings("ignore", category=UserWarning)
from copy import deepcopy
from collections import defaultdict
from typing import Any
import torch, gymnasium as gym
from sklearn.preprocessing import StandardScaler
from torchvision.transforms import v2 as transforms
from PIL import Image
import stable_worldmodel as swm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── run config ─────────────────────────────────────────────────────────────
EPISODE_IDX, START_STEP, GOAL_OFFSET = 2167, 50, 25
HORIZON, ACTION_BLOCK = 5, 5
NUM_SAMPLES, N_STEPS_CEM, TOPK, VAR_SCALE, SEED = 500, 30, 30, 1.0, 42
NEW_RUN_DIR = HACKIN / "push_t_book_3" / "runs" / f"ep{EPISODE_IDX}_s{START_STEP}_off{GOAL_OFFSET}_h{HORIZON}"
assert HORIZON * ACTION_BLOCK == GOAL_OFFSET, "plan length should equal goal offset"

# ── env + dataset + scaler + px/action calibration ─────────────────────────
ds = swm.data.HDF5Dataset(name="pusht_expert_train", keys_to_cache=["action", "proprio", "state"])
ep_starts = np.concatenate([[0], np.cumsum(ds.lengths)]).astype(np.int64)

all_actions = ds.get_col_data("action")
if all_actions.ndim == 3: all_actions = all_actions.reshape(-1, all_actions.shape[-1])
all_actions = all_actions[~np.isnan(all_actions).any(axis=1)]
action_scaler = StandardScaler().fit(all_actions)

states_raw, actions_raw = ds.get_col_data("state"), ds.get_col_data("action")
a_per_row = actions_raw.sum(axis=1) if actions_raw.ndim == 3 else actions_raw
r0, l0 = int(ep_starts[0]), int(ds.lengths[0])
xy, ac = states_raw[r0:r0+l0, :2].astype(float), a_per_row[r0:r0+l0].astype(float)
ok = ~(np.isnan(xy).any(1) | np.isnan(ac).any(1)); xy, ac = xy[ok], ac[ok]
dl, ac = np.diff(xy, axis=0), ac[:-1]
act_scale = np.array([(ac[:, 0] @ dl[:, 0]) / (ac[:, 0] @ ac[:, 0]),
                      (ac[:, 1] @ dl[:, 1]) / (ac[:, 1] @ ac[:, 1])])
IMG_SIZE_RUN = 224; SCALE = IMG_SIZE_RUN / WORLD
print("act_scale (px / unit):", act_scale.round(3))

img_transform = transforms.Compose([
    transforms.ToImage(), transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.Resize(size=IMG_SIZE_RUN),
])
cost_model = swm.policy.AutoCostModel("pusht/lewm").to(DEVICE).eval()
cost_model.requires_grad_(False); cost_model.interpolate_pos_encoding = True
CALLABLES = [{"method": "_set_state",      "args": {"state":      {"value": "state"}}},
             {"method": "_set_goal_state", "args": {"goal_state": {"value": "state"}}}]


In [ ]:
# ── policy / solver / world ────────────────────────────────────────────────
plan_config = swm.PlanConfig(horizon=HORIZON, receding_horizon=HORIZON, action_block=ACTION_BLOCK)
solver = swm.solver.CEMSolver(model=cost_model, batch_size=1, num_samples=NUM_SAMPLES,
                              var_scale=VAR_SCALE, n_steps=N_STEPS_CEM, topk=TOPK,
                              device=DEVICE, seed=SEED)
policy = swm.policy.WorldModelPolicy(solver=solver, config=plan_config,
                                     process={"action": action_scaler},
                                     transform={"pixels": img_transform, "goal": img_transform})
world = swm.World(env_name="swm/PushT-v1", num_envs=1, max_episode_steps=2 * GOAL_OFFSET,
                  history_size=1, frame_skip=1, image_shape=(IMG_SIZE_RUN, IMG_SIZE_RUN))
world.set_policy(policy)

def setup_world_for_episode(world, dataset, episode_idx, start_step, goal_offset, callables):
    """Replicates world.evaluate_from_dataset() for a single env (from p75d/p75bb)."""
    data = dataset.load_chunk(np.array([episode_idx]), np.array([start_step]), np.array([start_step + goal_offset]))
    init_per, goal_per = defaultdict(list), defaultdict(list)
    for ep in data:
        for col in dataset.column_names:
            if col.startswith("goal"): continue
            if col.startswith("pixels"): ep[col] = ep[col].permute(0, 2, 3, 1)
            if not isinstance(ep[col], (torch.Tensor, np.ndarray)): continue
            i0, g0 = ep[col][0], ep[col][-1]
            if isinstance(i0, torch.Tensor): i0, g0 = i0.numpy(), g0.numpy()
            init_per[col].append(i0); goal_per[col].append(g0)
    init_step = {k: np.stack(v) for k, v in deepcopy(init_per).items()}
    goal_step = {("goal" if k == "pixels" else f"goal_{k}"): np.stack(v) for k, v in goal_per.items()}
    variations = {k.removeprefix("variation."): v for k, v in init_step.items() if k.startswith("variation.")}
    options = [{} for _ in range(world.num_envs)]
    if variations:
        for i in range(world.num_envs):
            options[i]["variation"] = list(variations); options[i]["variation_values"] = {k: v[i] for k, v in variations.items()}
    reset_vals = deepcopy(init_step); reset_vals.update(deepcopy(goal_step))
    world.reset(seed=init_step.get("seed"), options=options)
    for i, env in enumerate(world.envs.unwrapped.envs):
        for spec in callables:
            m = getattr(env.unwrapped, spec["method"], None)
            if m is None: continue
            prepared = {}
            for name, ad in spec.get("args", {}).items():
                val = ad.get("value")
                if ad.get("in_dataset", True):
                    if val not in reset_vals: continue
                    prepared[name] = deepcopy(reset_vals[val][i])
                else: prepared[name] = val
            m(**prepared)
    shp = world.infos["pixels"].shape[:2]
    world.infos.update({k: np.broadcast_to(v[:, None], shp + v.shape[1:]) for k, v in init_step.items()})
    world.infos.update({k: np.broadcast_to(v[:, None], shp + v.shape[1:]) for k, v in goal_step.items()})
    return data, init_step, goal_step

data, init_step, goal_step = setup_world_for_episode(world, ds, EPISODE_IDX, START_STEP, GOAL_OFFSET, CALLABLES)
canvas   = world.envs.render()[0]
goal_img = goal_step["goal"][0]
row0 = ep_starts[EPISODE_IDX]
run_start_state = ds[row0 + START_STEP]["state"].squeeze(0).cpu().numpy()
run_goal_state  = ds[row0 + START_STEP + GOAL_OFFSET]["state"].squeeze(0).cpu().numpy()
agent_xy_img    = run_start_state[:2] * SCALE

fig, axs = plt.subplots(1, 2, figsize=(7, 3.5))
axs[0].imshow(canvas); axs[0].set_title("start"); axs[0].axis("off")
axs[1].imshow(goal_img); axs[1].set_title("goal"); axs[1].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# ── instrumented CEM + projection to image px + save ───────────────────────
def run_instrumented_cem(world, init_action=None):
    info_dict = world.policy._prepare_info(world.infos)
    s = world.policy.solver
    mean, var = s.init_action_distrib(init_action)
    mean, var = mean.to(s.device), var.to(s.device)
    hist = {k: [] for k in ["candidates", "costs", "topk_inds", "topk_candidates", "means", "vars"]}
    bs = s.n_envs; bm, bv = mean[:bs], var[:bs]
    expanded = {}
    for k, v in info_dict.items():
        vb = v[:bs]
        if torch.is_tensor(v): vb = vb.unsqueeze(1).expand(bs, s.num_samples, *vb.shape[1:])
        elif isinstance(v, np.ndarray): vb = np.repeat(vb[:, None, ...], s.num_samples, axis=1)
        expanded[k] = vb
    t0 = time.time()
    for step in range(s.n_steps):
        cands = torch.randn(bs, s.num_samples, s.horizon, s.action_dim, generator=s.torch_gen, device=s.device)
        cands = cands * bv.unsqueeze(1) + bm.unsqueeze(1); cands[:, 0] = bm
        costs = s.model.get_cost(expanded.copy(), cands)
        tv, ti = torch.topk(costs, k=s.topk, dim=1, largest=False)
        tc = cands[torch.arange(bs, device=s.device).unsqueeze(1).expand(-1, s.topk), ti]
        bm, bv = tc.mean(dim=1), tc.std(dim=1)
        for k, v in zip(hist, [cands, costs, ti, tc, bm, bv]): hist[k].append(v.detach().cpu())
    print(f"CEM: {time.time()-t0:.1f}s   final elite cost {tv.mean().item():.4f}")
    return hist

history = run_instrumented_cem(world)

N_ACTS_RUN, A_DIM = HORIZON * ACTION_BLOCK, 2
def candidates_to_image_xy(cands, start_xy_img):
    a = cands.detach().cpu().numpy().reshape(cands.shape[0], N_ACTS_RUN, A_DIM)
    a = action_scaler.inverse_transform(a.reshape(-1, A_DIM)).reshape(-1, N_ACTS_RUN, A_DIM)
    return np.cumsum(a * act_scale * SCALE, axis=1) + start_xy_img[None, None, :]

P_all, P_elite, P_mean, C, TK, CS = [], [], [], [], [], []
for it in range(N_STEPS_CEM):
    P_all.append(candidates_to_image_xy(history["candidates"][it][0], agent_xy_img))
    P_elite.append(candidates_to_image_xy(history["topk_candidates"][it][0], agent_xy_img))
    P_mean.append(candidates_to_image_xy(history["means"][it][0:1], agent_xy_img)[0])
    C.append(history["costs"][it][0].numpy()); TK.append(history["topk_inds"][it][0].numpy())
    sig = history["vars"][it][0].numpy().reshape(N_ACTS_RUN, A_DIM) * action_scaler.scale_ * act_scale * SCALE
    CS.append(np.sqrt(np.cumsum(sig ** 2, axis=0)))

NEW_RUN_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(NEW_RUN_DIR / "iterations.npz",
    paths_all=np.stack(P_all).astype(np.float32), paths_elite=np.stack(P_elite).astype(np.float32),
    paths_mean=np.stack(P_mean).astype(np.float32), costs=np.stack(C).astype(np.float32),
    topk_inds=np.stack(TK).astype(np.int32), cum_sigma=np.stack(CS).astype(np.float32),
    raw_means=torch.stack(history["means"]).numpy().astype(np.float32),
    raw_vars=torch.stack(history["vars"]).numpy().astype(np.float32))
json.dump({
    "episode_idx": EPISODE_IDX, "start_step": START_STEP, "goal_offset": GOAL_OFFSET,
    "horizon": HORIZON, "action_block": ACTION_BLOCK, "n_acts_per_plan": N_ACTS_RUN,
    "num_samples": NUM_SAMPLES, "n_steps": N_STEPS_CEM, "topk": TOPK, "var_scale": VAR_SCALE, "seed": SEED,
    "img_size": IMG_SIZE_RUN, "pusht_native": WORLD, "scale": float(SCALE),
    "agent_xy_img": agent_xy_img.tolist(), "start_state": run_start_state.tolist(), "goal_state": run_goal_state.tolist(),
    "act_scale": act_scale.tolist(), "action_scaler_mean": action_scaler.mean_.tolist(),
    "action_scaler_scale": action_scaler.scale_.tolist(),
}, open(NEW_RUN_DIR / "config.json", "w"), indent=2)
Image.fromarray(canvas).save(NEW_RUN_DIR / "start_img_raw.png")
Image.fromarray(goal_img).save(NEW_RUN_DIR / "goal_img_raw.png")
torch.save(history, NEW_RUN_DIR / "cem_history.pt")      # standardized candidates, for WM rollouts later
print("saved →", NEW_RUN_DIR, "\nset RUN_DIR to this and re-run from section 2")


## Scratch